# HireMinds AI - Resume & Job Description Matcher

**Run the cell below** (Shift+Enter), then upload your resume and job description.

In [2]:
# ============================================================
# HireMinds AI - Resume & Job Description Matcher
# Run this single cell to get the full interactive UI
# ============================================================
import os, io, sys
import ipywidgets as widgets
from IPython.display import display, Markdown

# Ensure project root is on the path
project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.pdf_parser import extract_text_from_pdf
from src.features import extract_and_build_features
from src.predict import predict_candidate


# ------ Helpers to extract bytes + filename from FileUpload ------
def _get_file_info(uploader):
    val = uploader.value
    if isinstance(val, dict):
        # ipywidgets >= 8: dict keyed by filename
        fname = next(iter(val))
        content = val[fname]['content']
    else:
        # ipywidgets < 8: tuple of Bunch
        item = val[0]
        content = getattr(item, 'content', None)
        fname = None
        meta = getattr(item, 'metadata', None)
        if isinstance(meta, dict):
            fname = meta.get('name')
        elif meta is not None:
            fname = getattr(meta, 'name', None)
    # Always convert to plain bytes (memoryview guard)
    if isinstance(content, memoryview):
        content = bytes(content)
    return content, fname


def load_resume_bytes(content_bytes, filename):
    if isinstance(content_bytes, memoryview):
        content_bytes = bytes(content_bytes)
    if filename:
        ext = os.path.splitext(filename)[1].lower()
    else:
        ext = '.pdf' if content_bytes[:4] == b'%PDF' else '.txt'
    if ext == '.pdf':
        tmp = os.path.join('data', 'resumes', '_uploaded_resume.pdf')
        os.makedirs(os.path.dirname(tmp), exist_ok=True)
        with open(tmp, 'wb') as f:
            f.write(content_bytes)
        return extract_text_from_pdf(tmp)
    else:
        return content_bytes.decode('utf-8', errors='ignore')


def load_job_bytes(content_bytes, filename):
    if isinstance(content_bytes, memoryview):
        content_bytes = bytes(content_bytes)
    return content_bytes.decode('utf-8', errors='ignore')


# ------ Upload widgets ------
resume_uploader = widgets.FileUpload(
    accept='.pdf,.txt', multiple=False, description='Upload Resume'
)
job_uploader = widgets.FileUpload(
    accept='.pdf', multiple=False, description='Upload Job Description'
)

# ------ Output area + button ------
output_area = widgets.Output()


def run_match(btn):
    output_area.clear_output()
    with output_area:
        if not resume_uploader.value or not job_uploader.value:
            print('Please upload both a resume and a job description.')
            return
        try:
            resume_bytes, resume_name = _get_file_info(resume_uploader)
            job_bytes, job_name = _get_file_info(job_uploader)
            resume_text = load_resume_bytes(resume_bytes, resume_name)
            job_text = load_job_bytes(job_bytes, job_name)
            features, details = extract_and_build_features(resume_text, job_text)
            prediction = predict_candidate(features)
            label = prediction['prediction']
            prob = prediction['probability'] * 100
            emoji = '\u2705' if label == 'SUITABLE' else '\u274c'
            matched = ', '.join(sorted(details['matched_skills'])) or 'None'
            missing = ', '.join(sorted(details['missing_skills'])) or 'None'
            candidate = ', '.join(sorted(details['candidate_skills'])) or 'None'
            required = ', '.join(sorted(details['required_skills'])) or 'None'
            jac = features['jaccard_similarity'] * 100
            cov = features['skill_coverage'] * 100
            exp_m = features['experience_match'] * 100
            edu = features['education_match'] * 100
            md = (
                f'## {emoji} Prediction: **{label}** ({prob:.1f}% confidence)\n\n'
                f'| Metric | Value |\n'
                f'|--------|-------|\n'
                f'| Jaccard Similarity | {jac:.1f}% |\n'
                f'| Skill Coverage | {cov:.1f}% |\n'
                f'| Experience Match | {exp_m:.1f}% |\n'
                f'| Education Match | {edu:.1f}% |\n\n'
                f'**\u2714 Matched Skills:** {matched}\n\n'
                f'**\u274c Missing Skills:** {missing}\n\n'
                f'**Candidate Skills:** {candidate}\n\n'
                f'**Required Skills:** {required}\n'
            )
            display(Markdown(md))
        except Exception as e:
            import traceback
            print('Error:', e)
            traceback.print_exc()


run_btn = widgets.Button(description='Run Match', button_style='success', icon='check')
run_btn.on_click(run_match)

# Display everything
display(
    widgets.VBox([
        widgets.HTML('<b>Step 1:</b> Upload your resume (PDF or TXT)'),
        resume_uploader,
        widgets.HTML('<b>Step 2:</b> Upload the job description (TXT)'),
        job_uploader,
        widgets.HTML('<b>Step 3:</b> Click Run Match'),
        run_btn,
        output_area
    ])
)
